In [1]:
from mpramnist.Gosai2024.dataset import GosaiDataset

from mpramnist.Chen2025.dataset import ChenMultiDataset
from mpramnist.Chen2025.dataset import ChenSingleDataset
from mpramnist.Chen2025.trainer import LitModel_Chen

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import mpramnist.transforms as t

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import lightning.pytorch as L
from lightning.pytorch.callbacks import ModelCheckpoint

from torchmetrics import PearsonCorrCoef

import pandas as pd

BATCH_SIZE = 1024
NUM_WORKERS = 8

I0000 00:00:1782550135.237463  923545 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782550136.116423  923545 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
import warnings
warnings.filterwarnings("ignore")


# Current Workflow

In this notebook, we:

1. Train the **MPRALegNet** model on the **Gosai SK-N-SH dataset**

2. Assess its predictive power using **Chen's SNPs**

## **Train MPRALegNet model using Gosai SK-N-SH data**

In [ ]:
cell_types = ["SKNSH"]

# preprocessing
train_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(0.5), t.Seq2Tensor(),])
val_test_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])

# load the data
train_dataset_own = GosaiDataset(
    split="train",
    transform=train_transform,
    filtration="own",
    cell_types=cell_types,
    stderr_columns=["SKNSH_lfcSE"], 
    stderr_threshold=1.0, 
    std_multiple_cut=6.0, 
    up_cutoff_move=3.0,
    duplication_cutoff=0.5, 
    root="../data",
)
# Use the same parameters to valid and test
val_dataset_own = GosaiDataset(split="val", filtration="own", cell_types=cell_types, stderr_columns=["SKNSH_lfcSE"], stderr_threshold=1.0, std_multiple_cut=6.0, up_cutoff_move=3.0, transform=val_test_transform, root='../data',)
test_dataset_own = GosaiDataset(split="test", filtration="own", cell_types=cell_types, stderr_columns=["SKNSH_lfcSE"], stderr_threshold=1.0, std_multiple_cut=6.0, up_cutoff_move=3.0, transform=val_test_transform, root='../data',)

print(train_dataset_own)

Dataset GosaiDataset (MpraDaraset)
    Number of datapoints: 842024
    Root location: /media/storage/lizzzafomenko/data/Malinois
    Using split: ['1', '2', '3', '4', '5', '6', '8', '9', '10', '11', '12', '14', '15', '16', '17', '18', '20', '22', 'Y']
    Split: {'train': 668946, 'val': 58809, 'test': 62582}
    Task: Regression
    Description: The Gosai dataset includes 798,064 sequences tested in the K562, HepG2, and SK-N-SH cell lines. The original sequence length is approximately 200 nucleotides, and it is recommended to extend them to 600 bp. The task is to predict three normalized regulatory activity values for the respective cell lines.


In [4]:
# encapsulate data into DataLoader form
train_loader = DataLoader(dataset=train_dataset_own, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(dataset=val_dataset_own, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset=test_dataset_own, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset_own[0][0])
out_channels = len(cell_types)

In [ ]:
# initialize LegNet model

model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Chen(
    model=model,
    loss=nn.MSELoss(),
    weight_decay=0.1,
    lr=0.01,
    print_each=1,
    use_one_cycle = True
)

In [6]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_pearson", mode="max", save_top_k=1, save_last=False
)

trainer = L.Trainer(
    accelerator="gpu",
    devices=[3],
    precision="16-mixed",
    enable_progress_bar=True,
    max_epochs=1,
    callbacks=[checkpoint_callback],
)

trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA L40S') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name          | Type            | Params | Mode  | FLOPs
------------------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | tra

Sanity Checking: |          | 0/? [00:00<?, ?it/s]


----------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 1.64248 | Val Pearson: -0.01054 | Train Pearson: nan 
----------------------------------------------------------------------------



Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...


-------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 0.56762 | Val Pearson: 0.84998 | Train Pearson: 0.72890 
-------------------------------------------------------------------------------



LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.4617306888103485
      test_pearson          0.8017808198928833
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.4617306888103485, 'test_pearson': 0.8017808198928833}]

In [ ]:
# load model from the best checkpoint

best_model_path = checkpoint_callback.best_model_path
seq_model = LitModel_Chen.load_from_checkpoint(best_model_path, model=model, loss=nn.MSELoss(), weight_decay=0.1, lr=0.01, print_each=1, use_one_cycle = True,)

## **Evaluate MPRALegNet model using Chen SNPs Data**

Initialize transformations for both `forward` and `reverse_complement` sequences.

For each sequence add flanks from Gosai assay and crop to the length of 600 base pairs.

In [5]:
forw_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])
revcomp_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(1), t.Seq2Tensor()])

#### **ChenMultiDataset**

In [ ]:
CELL_TYPES = ['Brain']
STATES = ['all']

In [ ]:
predict_forward_dataset = ChenMultiDataset(split = 'test', length = 227, cell_types = CELL_TYPES, states = STATES, transform=forw_transform, root = '../data')
predict_forward_dataloader = DataLoader(dataset=predict_forward_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

predict_revcomp_dataset = ChenMultiDataset(split = 'test', length = 227, cell_types = CELL_TYPES, states = STATES, transform=revcomp_transform, root = '../data')
predict_revcomp_dataloader = DataLoader(dataset=predict_revcomp_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [11]:
forw_preds = trainer.predict(seq_model, dataloaders=predict_forward_dataloader)
revcomp_preds = trainer.predict(seq_model, dataloaders=predict_revcomp_dataloader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [28]:
CELL_TYPE_MAPPING = {'THP1': ['aggregated', 'Naive', 'IFNB', 'IFNG', 'LPSIFNG'],
                    'HMC3': ['aggregated','Naive', 'IFNB', 'IFNG', 'LPSIFNG'],
                    'Brain': ['aggregated','Cortex', 'Hippocampus', 'Striatum']}


def Chen_variants_prediction(forw_preds, revcomp_preds, cell_types, return_df = True):
    """
    Calculate Pearson correlation between model-predicted and MPRA-measured 
    variant effects for each cell type.

    Parameters
    ----------
    forw_preds : list of dict
        List of dictionaries containing model predictions for forward sequences.
        Each dict must have keys: 'target', 'reverse_prediction', 'fdr', 'ref_predicted', 'alt_predicted'
    revcomp_preds : list of dict
        List of dictionaries containing model predictions for reverse complement sequences.
        Same structure as forw_preds
    cell_types : list of str
        List of cell type names corresponding to columns in target/fdr tensors
    return_df : bool
        If True, return results as pd.DataFrame; otherwise print them

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        - cell_type: name of the cell type
        - pearsonr_all: Pearson correlation across all tested variants
        - n_daSNP: number of significant variants (FDR < 0.05)
        - pearsonr_daSNP: Pearson correlation across significant variants only

    Note
    ----
    prediction_sign reverses effect (alt-ref to ref-alt) because MPRA variants logFC
    scores are calculated as log(Major/Minor). 
    """

    targets = torch.cat([pred["target"] for pred in forw_preds])
    fdrs = torch.cat([pred["fdr"] for pred in forw_preds])

    # prediction sign
    prediction_sign = torch.cat([pred["reverse_prediction"] for pred in forw_preds])

    y_preds_forw_ref = torch.cat([pred["ref_predicted"] for pred in forw_preds])
    y_preds_forw_alt = torch.cat([pred["alt_predicted"] for pred in forw_preds])


    y_preds_revcomp_ref = torch.cat([pred["ref_predicted"] for pred in revcomp_preds])
    y_preds_revcomp_alt = torch.cat([pred["alt_predicted"] for pred in revcomp_preds])

    y_preds_ref = torch.mean(torch.stack([y_preds_forw_ref, y_preds_revcomp_ref]), dim=0)
    y_preds_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_revcomp_alt]), dim=0)

    variant_prediction = (y_preds_alt - y_preds_ref).squeeze() * prediction_sign.squeeze()
    
    results = []
    pears = PearsonCorrCoef()

    for i in range(len(cell_types)):

        pearsonr_all = pears(variant_prediction.squeeze(), targets[:, i].squeeze())

        mask_daSNP = fdrs[:, i].squeeze() < 0.05     # remove non-significant variants with fdr > 0.05
        pearsonr_daSNP = pears(variant_prediction.squeeze()[mask_daSNP], targets[:, i].squeeze()[mask_daSNP])

        if return_df:
            results.append({
                'cell_type': cell_types[i],
                'n_all': len(variant_prediction),
                'pearsonr_all': pearsonr_all.item(),
                'n_daSNP': mask_daSNP.sum().item(),
                'pearsonr_daSNP': pearsonr_daSNP.item()
            })
    
        else:
            print(f'Pearson correlation for {cell_types[i]}')
            print(f'\t\t all SNPs (n = {len(variant_prediction)}): {pearsonr_all.item():.6f}')
            print(f'\t\t daSNPs (n = {mask_daSNP.sum().item()}): {pearsonr_daSNP.item():.6f}')

    if return_df:
        df = pd.DataFrame(results)
        return df


In [ ]:
Chen_variants_prediction(forw_preds, revcomp_preds, [f'Brain {state}' for state in CELL_TYPE_MAPPING['Brain']])

,cell_type,pearsonr_all,n_daSNP,pearsonr_daSNP
0,Brain aggregated,0.011953,186,0.022284
1,Brain Cortex,0.003440,13,0.300405
2,Brain Hippocampus,0.018114,4,0.030714
3,Brain Striatum,0.012880,27,0.028106


#### **ChenSingleDataset**

In [18]:
CELL_TYPE = 'Brain'
STATE = 'Cortex'

predict_forward_dataset = ChenSingleDataset(split = 'test', length = 227, cell_type = CELL_TYPE, state = STATE, interval_type = 'SNPCENTER', transform=forw_transform, root = '/media/storage/lizzzafomenko/data')
predict_forward_dataloader = DataLoader(dataset=predict_forward_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

predict_revcomp_dataset = ChenSingleDataset(split = 'test', length = 227, cell_type = CELL_TYPE, state = STATE, interval_type = 'SNPCENTER', transform=revcomp_transform, root = '/media/storage/lizzzafomenko/data')
predict_revcomp_dataloader = DataLoader(dataset=predict_revcomp_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [19]:
forw_preds = trainer.predict(seq_model, dataloaders=predict_forward_dataloader)
revcomp_preds = trainer.predict(seq_model, dataloaders=predict_revcomp_dataloader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [ ]:
Chen_variants_prediction(forw_preds, revcomp_preds, CELL_TYPE, False)

Pearson correlation for Brain
		 all SNPs (n = 222): 0.000000
		 daSNPs (n = 3): 0.358405


## Test AlphaGenome model

In [ ]:
from mpramnist.models import predict_variants_AlphaGenome, filter_tracks

I0000 00:00:1782314692.736602 3260483 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782314693.598890 3260483 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
# Use all available cell types
basic_transform = t.Compose([t.Seq2Tensor()])
ag_dataset = ChenMultiDataset(split = 'test', length = 2**11, transform=basic_transform, root = '/media/storage/lizzzafomenko/data')


In [5]:
ag_preds = predict_variants_AlphaGenome(weights_path = '/media/storage/lizzzafomenko/data/AlphaGenome/weights/model_all_folds.safetensors', 
                            dataset = ag_dataset, 
                            batch_size = 4, 
                            device = 'cuda:1') 

In [11]:
CT = ['THP1_aggregated', 'THP1_Naive', 'THP1_IFNB', 'THP1_IFNG', 'THP1_LPSIFNG', 'HMC3_aggregated', 'HMC3_Naive', 'HMC3_IFNB', 'HMC3_IFNG', 'HMC3_LPSIFNG', 'Brain_aggregated', 'Brain_Cortex', 'Brain_Hippocampus', 'Brain_Striatum']

In [17]:
CELL_TYPE_MAPPING = {'THP1': ['aggregated', 'Naive', 'IFNB', 'IFNG', 'LPSIFNG'],
                    'HMC3': ['aggregated','Naive', 'IFNB', 'IFNG', 'LPSIFNG'],
                    'Brain': ['aggregated','Cortex', 'Hippocampus', 'Striatum']}

def Chen_AlphaGenome_variants_prediction(preds, cell_types, return_df = True, biosample_names = None, exact_match = False):
    """ 
    Calculate Pearson correlation between model-predicted and MPRA-measured 
    variant effects for each cell type.

    Parameters
    ----------
    forw_preds : list of dict
        List of dictionaries containing model predictions for forward sequences.
        Each dict must have keys: 'target', 'reverse_prediction', 'fdr', 'ref_predicted', 'alt_predicted'
    revcomp_preds : list of dict
        List of dictionaries containing model predictions for reverse complement sequences.
        Same structure as forw_preds
    cell_types : list of str
        List of cell type names corresponding to columns in target/fdr tensors
    biosample_names : list[list[str]]
        List of biosample names to be used to filter AlphaGenome tracks 
        before variant prediction. If None, all tracks will be used for prediction
    return_df : bool
        If True, return results as pd.DataFrame; otherwise print them

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        - cell_type: name of the cell type
        - pearsonr_all: Pearson correlation across all tested variants
        - n_daSNP: number of significant variants (FDR < 0.05)
        - pearsonr_daSNP: Pearson correlation across significant variants only

    Note
    ----
    prediction_sign reverses effect (alt-ref to ref-alt) because MPRA variants logFC
    scores are calculated as log(Major/Minor). 
    """

    targets = torch.cat([pred["target"] for pred in preds])
    fdrs = torch.cat([pred["fdr"] for pred in preds])

    # prediction sign
    prediction_sign = torch.cat([pred["rev_pred"] for pred in preds])

    # sequence prediction
    y_preds_ref = torch.cat([pred["ref_predicted"] for pred in preds])
    y_preds_alt = torch.cat([pred["alt_predicted"] for pred in preds])

    variant_prediction = (y_preds_alt - y_preds_ref)
    
    results = []
    pears = PearsonCorrCoef()

    for i in range(len(cell_types)):

        if biosample_names:
            var_preds = filter_tracks(variant_prediction, biosample_names[i], exact_match = exact_match)
            var_preds = var_preds.mean(axis = 1)
        else:
            var_preds = variant_prediction.mean(axis = 1)

        var_preds = var_preds * prediction_sign

        pearsonr_all = pears(var_preds.squeeze(), targets[:, i].squeeze())

        mask_daSNP = fdrs[:, i].squeeze() < 0.05     # remove non-significant variants with fdr > 0.05
        pearsonr_daSNP = pears(var_preds.squeeze()[mask_daSNP], targets[:, i].squeeze()[mask_daSNP])

        if return_df:
            results.append({
                'cell_type': cell_types[i],
                'n_all': len(var_preds),
                'pearsonr_all': pearsonr_all.item(),
                'n_daSNP': mask_daSNP.sum().item(),
                'pearsonr_daSNP': pearsonr_daSNP.item()
            })
    
        else:
            print(f'Pearson correlation for {cell_types[i]}')
            print(f'\t\t all SNPs (n = {len(var_preds)}): {pearsonr_all.item():.6f}')
            print(f'\t\t daSNPs (n = {mask_daSNP.sum().item()}): {pearsonr_daSNP.item():.6f}')

    if return_df:
        df = pd.DataFrame(results)
        return df


In [19]:
Chen_AlphaGenome_variants_prediction(ag_preds, CT, True,
        [['macrophage'], ['macrophage'], ['macrophage'], ['inflammatory macrophage'], ['macrophage'], ['macrophage'], ['macrophage'], ['macrophage'], ['inflammatory macrophage'], ['macrophage'], ['brain'], ['frontal cortex', 'cerebellar cortex'], ['hippocampus'], ['striatum']],
        False)

,cell_type,n_all,pearsonr_all,n_daSNP,pearsonr_daSNP
0,THP1_aggregated,833,0.043900,343,0.056175
1,THP1_Naive,833,0.002080,223,0.066429
2,THP1_IFNB,833,0.092367,186,0.035688
3,THP1_IFNG,833,-0.042548,264,-0.060114
4,THP1_LPSIFNG,833,0.068838,190,0.070900
5,HMC3_aggregated,833,0.029073,112,0.040643
6,HMC3_Naive,833,0.026220,119,0.031110
7,HMC3_IFNB,833,0.023614,57,0.242006
8,HMC3_IFNG,833,0.002267,28,-0.079096
9,HMC3_LPSIFNG,833,0.013155,32,0.055986


In [ ]:
# use mean of all tracks

Chen_AlphaGenome_variants_prediction(ag_preds, CT, True)

,cell_type,n_all,pearsonr_all,n_daSNP,pearsonr_daSNP
0,THP1_aggregated,833,0.011627,343,0.037411
1,THP1_Naive,833,0.006182,223,0.007849
2,THP1_IFNB,833,0.037731,186,0.027283
3,THP1_IFNG,833,-0.055206,264,-0.103110
4,THP1_LPSIFNG,833,0.026061,190,0.044896
5,HMC3_aggregated,833,-0.025548,112,0.015918
6,HMC3_Naive,833,0.010214,119,0.008067
7,HMC3_IFNB,833,0.016433,57,0.174786
8,HMC3_IFNG,833,0.026881,28,-0.169554
9,HMC3_LPSIFNG,833,-0.010749,32,-0.044125


# DELETE ME

In [3]:
from mpramnist.models import BassetBranched, PARM, DREAM_RNN

In [6]:
predict_forward_dataset = ChenMultiDataset(split = 'test', length = 227, transform=forw_transform, root = '/media/storage/lizzzafomenko/data')
predict_forward_dataloader = DataLoader(dataset=predict_forward_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

predict_revcomp_dataset = ChenMultiDataset(split = 'test', length = 227, transform=revcomp_transform, root = '/media/storage/lizzzafomenko/data')
predict_revcomp_dataloader = DataLoader(dataset=predict_revcomp_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [19]:
trainer = L.Trainer(
    accelerator="cpu",
    #devices=[1],
    precision="16-mixed",
    enable_progress_bar=True,
    max_epochs=1,
)

/mnt/calc/lizzzafomenko/condaenvs/lizaim/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/accelerator_connector.py:479: You passed `Trainer(accelerator='cpu', precision='16-mixed')` but AMP with fp16 is not supported on CPU. Using `precision='bf16-mixed'` instead.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/mnt/calc/lizzzafomenko/condaenvs/lizaim/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
# load model from the best checkpoint

# 
# 
# 
best_model_path = checkpoint_callback.best_model_path

In [20]:
# initialize LegNet model

#model = BassetBranched(input_len=600, n_outputs=1)

# model = PARM(n_block=5, type_loss="mse", output_dim=1)

model = DREAM_RNN(in_channels=len(predict_forward_dataset[0][0]['seq']), seqsize=600, out_channels=1)

seq_model = LitModel_Chen(
    model=model,
    loss=nn.MSELoss(),
    weight_decay=0.1,
    lr=0.01,
    print_each=1,
    use_one_cycle = True
)

In [ ]:

Malinois

/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/lightning_logs/version_10/checkpoints/epoch=23-step=19752.ckpt

/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/lightning_logs/version_11/checkpoints/epoch=8-step=7407.ckpt

/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/lightning_logs/version_12/checkpoints/epoch=3-step=3292.ckpt

/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/lightning_logs/version_13/checkpoints/epoch=18-step=15637.ckpt

/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/lightning_logs/version_14/checkpoints/epoch=8-step=7407.ckpt

SyntaxError: invalid decimal literal (3439104669.py, line 1)

In [ ]:
PARM

/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/lightning_logs/version_15/checkpoints/epoch=41-step=34566.ckpt


/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/lightning_logs/version_17/checkpoints/epoch=40-step=33743.ckpt

/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/lightning_logs/version_18/checkpoints/epoch=45-step=37858.ckpt

/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/lightning_logs/version_19/checkpoints/epoch=42-step=35389.ckpt

/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/lightning_logs/version_20/checkpoints/epoch=41-step=34566.ckpt

In [21]:
# DREAM-RNN

chekpoints = [
'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Gosai2024/lightning_logs/version_0/checkpoints/epoch=42-step=35389.ckpt',

'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Gosai2024/lightning_logs/version_1/checkpoints/epoch=47-step=39504.ckpt',

'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Gosai2024/lightning_logs/version_2/checkpoints/epoch=0-step=823.ckpt',

'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Gosai2024/lightning_logs/version_3/checkpoints/epoch=42-step=35389.ckpt',

'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Gosai2024/lightning_logs/version_4/checkpoints/epoch=47-step=39504.ckpt',
]

In [23]:
forw_preds[0]['ref_predicted'].shape

torch.Size([833, 1])

In [29]:
for i, ckp in enumerate(chekpoints):
    seq_model = LitModel_Chen.load_from_checkpoint(ckp, model=model, loss=nn.MSELoss(), weight_decay=0.1, lr=0.01, print_each=1, use_one_cycle = True,)
    forw_preds = trainer.predict(seq_model, dataloaders=predict_forward_dataloader)
    revcomp_preds = trainer.predict(seq_model, dataloaders=predict_revcomp_dataloader)
    df = Chen_variants_prediction(forw_preds, revcomp_preds, CT, True)
    df.to_csv(f'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Chen2025/results/DREAM_RNN_run{i}.tsv', sep='\t', index = False)

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()